# Danbooru Interactive Dashboard

Interactive exploration notebook for `data/parquet/danbooru.parquet`.

Included views:
- rating, file extension, and status distributions
- scatter plots for score, favorites, tags, and image size
- top tags by tag category
- monthly upload trends
- correlation heatmap for numeric features


In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

pd.set_option('display.max_columns', 100)
px.defaults.template = 'plotly_white'

def _configure_plotly_renderer():
    # Plotly inline rendering in Jupyter requires nbformat.
    # If it's missing, fall back to opening plots in the browser.
    try:
        import nbformat  # noqa: F401
        pio.renderers.default = 'notebook_connected'
    except Exception:
        pio.renderers.default = 'browser'
        print("Plotly inline rendering requires 'nbformat'. Using browser renderer instead.")
        print("To enable inline plots: pip install nbformat  (or: conda install nbformat)")

_configure_plotly_renderer()

DATA_PATH = Path.cwd().parent / 'data' / 'parquet' / 'danbooru.parquet'
SAMPLE_SIZE = 150_000

lazy_df = pl.scan_parquet(DATA_PATH)
sample_df = lazy_df.head(SAMPLE_SIZE).collect().to_pandas()

sample_df['created_at'] = pd.to_datetime(sample_df['created_at'], errors='coerce')
sample_df['rating'] = sample_df['rating'].fillna('unknown')
sample_df['file_ext'] = sample_df['file_ext'].fillna('unknown')
sample_df['source'] = sample_df['source'].fillna('')
sample_df['has_source'] = sample_df['source'].str.strip().ne('')
sample_df['pixiv_id'] = pd.to_numeric(sample_df['pixiv_id'], errors='coerce')
sample_df['parent_id'] = pd.to_numeric(sample_df['parent_id'], errors='coerce')
sample_df['tag_count'] = pd.to_numeric(sample_df['tag_count'], errors='coerce')
sample_df['score'] = pd.to_numeric(sample_df['score'], errors='coerce')
sample_df['fav_count'] = pd.to_numeric(sample_df['fav_count'], errors='coerce')
sample_df['image_width'] = pd.to_numeric(sample_df['image_width'], errors='coerce')
sample_df['image_height'] = pd.to_numeric(sample_df['image_height'], errors='coerce')
sample_df['file_size'] = pd.to_numeric(sample_df['file_size'], errors='coerce')
sample_df['aspect_ratio'] = sample_df['image_width'] / sample_df['image_height'].replace(0, np.nan)
sample_df['megapixels'] = (sample_df['image_width'] * sample_df['image_height']) / 1_000_000
sample_df['log_file_size_mb'] = np.log10(sample_df['file_size'].clip(lower=1) / (1024 * 1024))
sample_df['year_month'] = sample_df['created_at'].dt.to_period('M').dt.to_timestamp()

numeric_columns = [
    'score', 'up_score', 'down_score', 'fav_count', 'image_width', 'image_height',
    'file_size', 'tag_count', 'aspect_ratio', 'megapixels', 'log_file_size_mb'
]

print(f'Data file: {DATA_PATH}')
print(f'Sampled rows: {len(sample_df):,}')
display(sample_df.head())


Plotly inline rendering requires 'nbformat'. Using browser renderer instead.
To enable inline plots: pip install nbformat  (or: conda install nbformat)
Data file: c:\Users\EL069\Project\safebooru\data\parquet\danbooru.parquet
Sampled rows: 150,000


C:\Users\EL069\AppData\Local\Temp\ipykernel_17000\691983830.py:51: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  sample_df['year_month'] = sample_df['created_at'].dt.to_period('M').dt.to_timestamp()


,post_id,created_at,rating,score,up_score,down_score,fav_count,md5,file_ext,image_width,image_height,file_size,tag_string,tag_count,tag_string_general,tag_string_character,tag_string_copyright,tag_string_artist,tag_string_meta,parent_id,has_children,source,pixiv_id,is_deleted,is_banned,has_source,aspect_ratio,megapixels,log_file_size_mb,year_month
0,11166498,2026-04-15 06:55:20.799000-04:00,s,0,0,0,0,3ec5ec2212bb41540a5175bb9861f408,jpg,2480,3508,2833585,1girl absurdres bikini black_bikini black_chok...,33,1girl bikini black_bikini black_choker black_r...,sakuraba_ema,mahou_shoujo_no_majo_saiban,icedream,absurdres highres,NaN,False,https://i.pximg.net/img-original/img/2026/02/2...,141696017.0,False,False,True,0.706956,8.699840,0.431736,2026-04-01
1,11166497,2026-04-15 06:55:07.524000-04:00,s,0,0,0,0,20e8d33953a57508894fcf1726f1e889,jpg,1992,3110,2170028,1girl absurdres alternate_costume bloomers blu...,25,1girl alternate_costume bloomers blue_eyes blu...,chrono_genesis_(umamusume),umamusume,siomi_n,absurdres highres,NaN,False,https://i.pximg.net/img-original/img/2026/04/1...,143561630.0,False,False,True,0.640514,6.195120,0.315865,2026-04-01
2,11166496,2026-04-15 06:54:41.748000-04:00,g,0,0,0,0,5b50e5d3790c40b5c546dc664d929caf,jpg,2480,3508,2837800,2girls absurdres black_dress black_hair black_...,43,2girls black_dress black_hair black_hat blush ...,nikaido_hiro sakuraba_ema,mahou_shoujo_no_majo_saiban,icedream,absurdres highres,NaN,False,https://i.pximg.net/img-original/img/2026/03/0...,141913161.0,False,False,True,0.706956,8.699840,0.432382,2026-04-01
3,11166495,2026-04-15 06:54:39.292000-04:00,g,1,1,0,0,7617239c82cca9bee05561b5158b1019,jpg,1271,858,145131,1girl :d ^_^ animal_print artist_name bell blu...,40,1girl :d ^_^ animal_print artist_name bell blu...,tachibana_chitose,fatal_frame fatal_frame_ii:_crimson_butterfly,oisiso,commentary,NaN,False,https://x.com/tetewujin/status/204434020084820...,NaN,False,False,True,1.481352,1.090518,-0.858840,2026-04-01
4,11166494,2026-04-15 06:54:13.884000-04:00,s,1,1,0,1,d0d8e28e512c97c20d4208426f4a3a8b,jpg,4160,5579,4064686,1girl absurdres arist_name ass bare_shoulders ...,38,1girl arist_name ass bare_shoulders black_boot...,cissia_(zenless_zone_zero),zenless_zone_zero,ringeko-chan,absurdres commentary english_commentary highres,NaN,True,https://i.pximg.net/img-original/img/2026/04/1...,143588014.0,False,False,True,0.745653,23.208640,0.588427,2026-04-01


In [2]:
schema_df = pd.DataFrame({
    'column': sample_df.columns,
    'dtype': [str(sample_df[col].dtype) for col in sample_df.columns],
    'null_count': [int(sample_df[col].isna().sum()) for col in sample_df.columns],
})
schema_df


,column,dtype,null_count
0,post_id,int64,0
1,created_at,"datetime64[ns, UTC-04:00]",0
2,rating,object,0
3,score,int64,0
4,up_score,int64,0
5,down_score,int64,0
6,fav_count,int64,0
7,md5,object,0
8,file_ext,object,0
9,image_width,int64,0


In [3]:
summary_metrics = pd.DataFrame({
    'metric': ['Rows', 'Unique tags', 'Avg score', 'Avg favorites', 'Avg megapixels', 'Pixiv-linked posts'],
    'value': [
        len(sample_df),
        sample_df['tag_string'].fillna('').str.split().explode().nunique(),
        sample_df['score'].mean(),
        sample_df['fav_count'].mean(),
        sample_df['megapixels'].mean(),
        sample_df['pixiv_id'].notna().sum(),
    ]
})
summary_metrics['value'] = summary_metrics['value'].map(lambda x: f'{x:,.2f}')
display(summary_metrics)

rating_counts = sample_df['rating'].value_counts().rename_axis('rating').reset_index(name='count')
ext_counts = sample_df['file_ext'].value_counts().head(10).rename_axis('file_ext').reset_index(name='count')
status_counts = pd.DataFrame({
    'status': ['active', 'deleted', 'banned'],
    'count': [
        int((~sample_df['is_deleted'].fillna(False) & ~sample_df['is_banned'].fillna(False)).sum()),
        int(sample_df['is_deleted'].fillna(False).sum()),
        int(sample_df['is_banned'].fillna(False).sum()),
    ]
})

fig = px.bar(rating_counts, x='rating', y='count', color='rating', title='Rating Distribution')
fig.update_layout(showlegend=False)
fig.show()

fig = px.treemap(ext_counts, path=['file_ext'], values='count', title='Top File Extensions')
fig.show()

fig = px.pie(status_counts, names='status', values='count', hole=0.45, title='Post Availability Status')
fig.show()


,metric,value
0,Rows,"150,000.00"
1,Unique tags,"117,352.00"
2,Avg score,12.40
3,Avg favorites,11.57
4,Avg megapixels,5.30
5,Pixiv-linked posts,"58,089.00"


In [4]:
scatter_columns = ['score', 'fav_count', 'tag_count', 'image_width', 'image_height', 'file_size', 'megapixels', 'aspect_ratio']
scatter_sample = sample_df.dropna(subset=['rating']).copy()

def render_scatter(x_axis, y_axis, color_by, rating_filter, max_points):
    filtered = scatter_sample.copy()
    if rating_filter != 'all':
        filtered = filtered[filtered['rating'] == rating_filter]
    filtered = filtered.dropna(subset=[x_axis, y_axis]).head(max_points)

    fig = px.scatter(
        filtered,
        x=x_axis,
        y=y_axis,
        color=color_by,
        hover_data=['post_id', 'rating', 'file_ext', 'tag_count', 'fav_count'],
        title=f'{y_axis} vs {x_axis}',
        opacity=0.65,
    )
    fig.update_traces(marker={'size': 7})
    fig.show()

widgets.interact(
    render_scatter,
    x_axis=widgets.Dropdown(options=scatter_columns, value='score', description='X'),
    y_axis=widgets.Dropdown(options=scatter_columns, value='fav_count', description='Y'),
    color_by=widgets.Dropdown(options=['rating', 'file_ext', 'is_deleted', 'is_banned', 'has_source'], value='rating', description='Color'),
    rating_filter=widgets.Dropdown(options=['all'] + sorted(sample_df['rating'].dropna().unique().tolist()), value='all', description='Rating'),
    max_points=widgets.IntSlider(value=20000, min=2000, max=50000, step=2000, description='Points'),
);


interactive(children=(Dropdown(description='X', options=('score', 'fav_count', 'tag_count', 'image_width', 'im…

In [5]:
tag_columns = {
    'all tags': 'tag_string',
    'general': 'tag_string_general',
    'character': 'tag_string_character',
    'copyright': 'tag_string_copyright',
    'artist': 'tag_string_artist',
    'meta': 'tag_string_meta',
}

def render_top_tags(tag_family, rating_filter, top_n):
    column = tag_columns[tag_family]
    filtered = sample_df.copy()
    if rating_filter != 'all':
        filtered = filtered[filtered['rating'] == rating_filter]

    tag_series = filtered[column].fillna('').astype(str)
    tag_counter = Counter(' '.join(tag_series).split())
    top_tags = pd.DataFrame(tag_counter.most_common(top_n), columns=['tag', 'count'])

    fig = px.bar(
        top_tags.sort_values('count'),
        x='count',
        y='tag',
        orientation='h',
        title=f'Top {top_n} {tag_family} tags',
        color='count',
        color_continuous_scale='Sunsetdark',
    )
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})
    fig.show()

widgets.interact(
    render_top_tags,
    tag_family=widgets.Dropdown(options=list(tag_columns.keys()), value='all tags', description='Tag set'),
    rating_filter=widgets.Dropdown(options=['all'] + sorted(sample_df['rating'].dropna().unique().tolist()), value='all', description='Rating'),
    top_n=widgets.IntSlider(value=25, min=10, max=50, step=5, description='Top N'),
);


interactive(children=(Dropdown(description='Tag set', options=('all tags', 'general', 'character', 'copyright'…

In [6]:
monthly_counts = sample_df.dropna(subset=['year_month']).groupby(['year_month', 'rating']).size().reset_index(name='count')

def render_monthly_trend(metric, rating_mode):
    filtered = monthly_counts.copy()
    if rating_mode != 'all':
        filtered = filtered[filtered['rating'] == rating_mode]

    if metric == 'post_count':
        fig = px.line(filtered, x='year_month', y='count', color='rating', markers=True, title='Monthly Post Count Trend')
    else:
        metric_df = sample_df.dropna(subset=['year_month']).groupby(['year_month', 'rating'])[metric].mean().reset_index()
        if rating_mode != 'all':
            metric_df = metric_df[metric_df['rating'] == rating_mode]
        fig = px.line(metric_df, x='year_month', y=metric, color='rating', markers=True, title=f'Monthly Average {metric}')
    fig.show()

widgets.interact(
    render_monthly_trend,
    metric=widgets.Dropdown(options=['post_count', 'score', 'fav_count', 'tag_count', 'megapixels'], value='post_count', description='Metric'),
    rating_mode=widgets.Dropdown(options=['all'] + sorted(sample_df['rating'].dropna().unique().tolist()), value='all', description='Rating'),
);


interactive(children=(Dropdown(description='Metric', options=('post_count', 'score', 'fav_count', 'tag_count',…

In [7]:
corr_df = sample_df[numeric_columns].corr(numeric_only=True).round(2)
fig = px.imshow(
    corr_df,
    text_auto=True,
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Correlation Heatmap for Numeric Features',
)
fig.show()

source_summary = pd.DataFrame({
    'category': ['Has source', 'No source', 'Pixiv-linked', 'Has parent'],
    'count': [
        int(sample_df['has_source'].sum()),
        int((~sample_df['has_source']).sum()),
        int(sample_df['pixiv_id'].notna().sum()),
        int(sample_df['parent_id'].notna().sum()),
    ]
})

fig = px.funnel(source_summary, x='count', y='category', title='Source / Relationship Coverage')
fig.show()
